In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import time
import os

In [ ]:
current_dir = Path.cwd().parent

In [ ]:
train_dir = str(current_dir / 'datasets/Train/train.csv')
valid_dir = str(current_dir / 'datasets/Validation/valid.csv')
test_dir = str(current_dir / 'datasets/Test/test.csv')
g18_19_dir = str(current_dir / 'datasets/GH18_GH19_descriptores.csv')
emb_dir = str(current_dir / 'datasets/All_Embeddings')
valid_emb_dir = str(current_dir / 'datasets/Valid_Embeddings')
test_emb_dir = str(current_dir / 'datasets/Test_Embeddings')
g18_19_emb_dir = str(current_dir / 'datasets/G18_G19_Embeddings')

In [ ]:
info = np.load(os.path.join(emb_dir, 'info.npy'))
info[0] = 8
np.save(os.path.join(emb_dir, 'info.npy'), info)

In [16]:
df_train = pd.read_csv(train_dir)
df_train.head(5)

,id_uniprot,sequence,ec_number,IsEnzime
0,C0HLW5,TDAVANGVHAISGVVDS,0.0.0.0,0
1,Q66C72,MAQVAKKILVTCALPYANGSIHLGHMLEHIQADIWVRFQRMRGNQV...,6.1.1.10,1
2,Q3Z4P3,MATLFIADLHLCVEEPAITAGFLRFLAGEARKADALYILGDLFEAW...,3.6.1.54,1
3,Q6YDN7,MLRRKPTRLELKLDDIEEFENIRKDLEARKKQKEDVEGVGTSDGEG...,0.0.0.0,0
4,Q1JHB8,MEALKKIAGVTAAQYVTDGMTIGLGTGSTAYYFVEEIGRRVKQEGL...,5.3.1.6,1


In [ ]:
len(df_train)

396832

In [ ]:
df_valid = pd.read_csv(valid_dir)
df_valid.head(5)

,id_uniprot,sequence,ec_number,IsEnzime
0,A7NR45,MSKLKITYRKSAIGYSHDQKATIRSLGLRRLNSVVVHDDTPTIRGM...,0.0.0.0,0
1,Q5FM74,MANRNDSRNNRRNKDDIEDQLVAVNRITKVVKGGRRMRFAALVVVG...,0.0.0.0,0
2,Q68WM5,MIKKEIIILCGPTASGKSYLGHELAKAYGCEIINIDSMQVYREIPI...,2.5.1.75,1
3,Q6HJF2,MKKVDVKESAVGRETRIRKQWNEQSIFEQSIQNREGAQSFVFYEGP...,6.1.1.5,1
4,Q35438,MTNIRKTHPLMKIINSSFIDLPAPSNISAWWNFGSLLGICLILQIL...,0.0.0.0,0


In [ ]:
df_test = pd.read_csv(test_dir)
df_test.head(5)

,Unnamed: 0,id_uniprot,sequence,ec_number,IsEnzime
0,0,Q03599,MQWPLIAALFTQLYHFVIGDDCYCPLCELDKLPDNCTTIHGHIQMG...,0.0.0.0,0
1,1,P0CH32,MQIFVKTLTGKTITLEVESSDTIDNVKAKIQDKEGIPPDQQRLIFA...,0.0.0.0,0
2,2,Q41224,MMTRGATRMTRTVLGHMGPRYFSTAIFRNDAGTGVMSGAAVFMHGV...,1.10.3.11,1
3,3,Q6CUC6,MAKSPKRKNSEPEDLSSNKRSKSFEFGKVENGLEPSGSDFGSDLPT...,0.0.0.0,0
4,4,Q9TF49,MTNTRKTHPLIKIINHSFIDLPAPSNISAWWNFGSLLGLCLIIQIL...,0.0.0.0,0


In [4]:
df_g18_19 = pd.read_csv(g18_19_dir)
df_g18_19.head(5)

,Nombre,Secuencia,Familia,A,R,N,D,C,E,Q,...,_HydrophobicityD2001,_HydrophobicityD2025,_HydrophobicityD2050,_HydrophobicityD2075,_HydrophobicityD2100,_HydrophobicityD3001,_HydrophobicityD3025,_HydrophobicityD3050,_HydrophobicityD3075,_HydrophobicityD3100
0,AAB97779.1,MRKFNKPLLALLIGSTLCSAAQAAAPGKPTIAWGNTKFAIVEVDQA...,GH18,9.947,2.309,6.039,6.572,0.888,3.375,4.618,...,1.243,22.202,49.378,77.087,99.645,0.178,25.755,49.201,71.226,99.822
1,AAC44673.1,EYHMGNTMTKAIYDKFKAATPYGNTVATGAVPSETVDIAVSIGGFK...,GH18,9.639,1.205,6.627,7.831,0.000,2.410,2.410,...,1.205,19.880,49.398,72.289,99.398,2.410,25.301,48.193,74.699,97.590
2,AAM08773.1,MVNGYLFREYIGAQFTGVRFSDVPVNPGLSFHFILAFAIDYFMATQ...,GH18,13.287,3.147,4.895,5.245,0.350,3.846,3.846,...,1.399,24.126,46.853,72.727,100.000,0.350,19.930,47.902,71.678,98.951
3,AAM08776.1,MTNGYLFREYIGAQFTGVRFSDVPINPNLSFNFILSFAIDYTSPAG...,GH18,10.417,2.778,4.861,7.292,0.347,3.819,3.472,...,0.694,21.875,44.097,72.917,100.000,0.347,24.653,49.653,71.875,98.958
4,BAB79618.1,MKQHVKTLKLSQMASILGAIFCANSYAYNCDGLPEWSQSSAYNGGA...,GH18,8.582,1.639,8.100,6.654,0.868,3.761,3.954,...,0.386,25.169,49.952,76.374,99.711,0.096,24.108,50.916,74.542,100.000


In [5]:
!pip install "transformers<5"
import transformers
from transformers import BertTokenizer, TFBertModel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


In [6]:
model_name = "Rostlab/prot_bert"
tokenizer = BertTokenizer.from_pretrained(model_name, do_lower_case=False)
model = TFBertModel.from_pretrained(model_name, from_pt=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForS

In [ ]:
df_train['processed_sequence'] = df_train['sequence'].apply(lambda seq: " ".join(list(seq)))

In [ ]:
df_valid['processed_sequence'] = df_valid['sequence'].apply(lambda seq: " ".join(list(seq)))

In [ ]:
df_test['processed_sequence'] = df_test['sequence'].apply(lambda seq: " ".join(list(seq)))

In [7]:
df_g18_19['processed_sequence'] = df_g18_19['Secuencia'].apply(lambda seq: " ".join(list(seq)))

In [8]:
length_subsets = 40000

In [ ]:
def generate_embeddings(sequences, dir, batch_size=96):
    global length_subsets, tokenizer, model
    def generate_embeddings_numpy(sequences, batch_size):
        global tokenizer, model
        all_embeddings = []
        for i in range(0, len(sequences), batch_size):
            end = len(sequences) if i + batch_size > len(sequences) else i + batch_size
            print(f'De {i} a {end}')
            batch_seqs = sequences[i:end]
            encoded = tokenizer(
                batch_seqs.tolist(),
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="tf"
            )
            outputs = model(encoded)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
            all_embeddings.append(cls_embeddings)
        return np.vstack(all_embeddings)

    info_name = 'info.npy'
    info_dir = os.path.join(dir, info_name)
    if not os.path.exists(info_dir):
        data = np.array([0, 0])
        np.save(info_dir, data)
    data = np.load(info_dir)
    current_subset = int(data[0])
    current_time = float(data[1])
    total_subsets = len(sequences) // length_subsets if len(sequences) % length_subsets == 0 else len(sequences) // length_subsets + 1
    if current_subset == total_subsets:
        print('Embeddings already generated')
        return
    for i in range(int(current_subset), int(total_subsets)):
        print(f'Starting subset: {i} current time: {current_time}s')
        start = time.time()
        start_index = length_subsets * current_subset
        end_index = min(start_index + length_subsets, len(sequences))
        embs = generate_embeddings_numpy(sequences[start_index:end_index], batch_size)
        np.save(os.path.join(dir, f'subset_{current_subset}.npy'), embs)
        current_subset += 1
        end = time.time()
        current_time += end - start
        np.save(info_dir, np.array([current_subset, current_time]))
        print(f'Saved, current time: {current_time}s')

In [ ]:
generate_embeddings(df_train['processed_sequence'], emb_dir)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Starting subset: 8 current time: 36374.7127468586s
De 0 a 96
De 96 a 192
De 192 a 288
De 288 a 384
De 384 a 480
De 480 a 576
De 576 a 672
De 672 a 768
De 768 a 864
De 864 a 960
De 960 a 1056
De 1056 a 1152
De 1152 a 1248
De 1248 a 1344
De 1344 a 1440
De 1440 a 1536
De 1536 a 1632
De 1632 a 1728
De 1728 a 1824
De 1824 a 1920
De 1920 a 2016
De 2016 a 2112
De 2112 a 2208
De 2208 a 2304
De 2304 a 2400
De 2400 a 2496
De 2496 a 2592
De 2592 a 2688
De 2688 a 2784
De 2784 a 2880
De 2880 a 2976
De 2976 a 3072
De 3072 a 3168
De 3168 a 3264
De 3264 a 3360
De 3360 a 3456
De 3456 a 3552
De 3552 a 3648
De 3648 a 3744
De 3744 a 3840
De 3840 a 3936
De 3936 a 4032
De 4032 a 4128
De 4128 a 4224
De 4224 a 4320
De 4320 a 4416
De 4416 a 4512
De 4512 a 4608
De 4608 a 4704
De 4704 a 4800
De 4800 a 4896
De 4896 a 4992
De 4992 a 5088
De 5088 a 5184
De 5184 a 5280
De 5280 a 5376
De 5376 a 5472
De 5472 a 5568
De 5568 a 5664
De 5664 a 5760
De 5760 a 5856
De 5856 a 5952
De 5952 a 6048
De 6048 a 6144
De 6144 a 6240

In [ ]:
generate_embeddings(df_valid['processed_sequence'], valid_emb_dir)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Starting subset: 0 current time: 0.0s
De 0 a 96
De 96 a 192
De 192 a 288
De 288 a 384
De 384 a 480
De 480 a 576
De 576 a 672
De 672 a 768
De 768 a 864
De 864 a 960
De 960 a 1056
De 1056 a 1152
De 1152 a 1248
De 1248 a 1344
De 1344 a 1440
De 1440 a 1536
De 1536 a 1632
De 1632 a 1728
De 1728 a 1824
De 1824 a 1920
De 1920 a 2016
De 2016 a 2112
De 2112 a 2208
De 2208 a 2304
De 2304 a 2400
De 2400 a 2496
De 2496 a 2592
De 2592 a 2688
De 2688 a 2784
De 2784 a 2880
De 2880 a 2976
De 2976 a 3072
De 3072 a 3168
De 3168 a 3264
De 3264 a 3360
De 3360 a 3456
De 3456 a 3552
De 3552 a 3648
De 3648 a 3744
De 3744 a 3840
De 3840 a 3936
De 3936 a 4032
De 4032 a 4128
De 4128 a 4224
De 4224 a 4320
De 4320 a 4416
De 4416 a 4512
De 4512 a 4608
De 4608 a 4704
De 4704 a 4800
De 4800 a 4896
De 4896 a 4992
De 4992 a 5088
De 5088 a 5184
De 5184 a 5280
De 5280 a 5376
De 5376 a 5472
De 5472 a 5568
De 5568 a 5664
De 5664 a 5760
De 5760 a 5856
De 5856 a 5952
De 5952 a 6048
De 6048 a 6144
De 6144 a 6240
De 6240 a 63

In [ ]:
generate_embeddings(df_test['processed_sequence'], test_emb_dir)

Starting subset: 0 current time: 0.0s
De 0 a 96
De 96 a 192
De 192 a 288
De 288 a 384
De 384 a 480
De 480 a 576
De 576 a 672
De 672 a 768
De 768 a 864
De 864 a 960
De 960 a 1056
De 1056 a 1152
De 1152 a 1248
De 1248 a 1344
De 1344 a 1440
De 1440 a 1536
De 1536 a 1632
De 1632 a 1728
De 1728 a 1824
De 1824 a 1920
De 1920 a 2016
De 2016 a 2112
De 2112 a 2208
De 2208 a 2304
De 2304 a 2400
De 2400 a 2496
De 2496 a 2592
De 2592 a 2688
De 2688 a 2784
De 2784 a 2880
De 2880 a 2976
De 2976 a 3072
De 3072 a 3168
De 3168 a 3264
De 3264 a 3360
De 3360 a 3456
De 3456 a 3552
De 3552 a 3648
De 3648 a 3744
De 3744 a 3840
De 3840 a 3936
De 3936 a 4032
De 4032 a 4128
De 4128 a 4224
De 4224 a 4320
De 4320 a 4416
De 4416 a 4512
De 4512 a 4608
De 4608 a 4704
De 4704 a 4800
De 4800 a 4896
De 4896 a 4992
De 4992 a 5088
De 5088 a 5184
De 5184 a 5280
De 5280 a 5376
De 5376 a 5472
De 5472 a 5568
De 5568 a 5664
De 5664 a 5760
De 5760 a 5856
De 5856 a 5952
De 5952 a 6048
De 6048 a 6144
De 6144 a 6240
De 6240 a 63

In [12]:
generate_embeddings(df_g18_19['processed_sequence'], g18_19_emb_dir)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Starting subset: 0 current time: 0.0s
De 0 a 96
De 96 a 192
De 192 a 288
De 288 a 384
De 384 a 459
Saved, current time: 75.42132997512817s


In [19]:
embs_train = np.array([])
for i in range((len(df_train) // 40000) + 1):
  subset = np.load(os.path.join(emb_dir, f'subset_{i}.npy'))
  if len(embs_train) == 0:
    embs_train = subset
  else:
    embs_train = np.vstack((embs_train, subset))

In [ ]:
np.save(str(current_dir / 'datasets/First Level/train_embeddings.npy'), embs_train)

In [ ]:
valid_embeddings = np.load(str(current_dir / 'datasets/Validation/valid_embeddings/subset_0.npy'))
np.save(str(current_dir / 'datasets/First Level/valid_embeddings.npy'), valid_embeddings)
test_embeddings = np.load(str(current_dir / 'datasets/Test/test_embeddings/subset_0.npy'))
np.save(str(current_dir / 'datasets/First Level/test_embeddings.npy'), test_embeddings)


In [ ]:
g18_g19_embeddings = np.load(str(current_dir / 'datasets/G18_G19_Embeddings/subset_0.npy'))

In [ ]:
np.save(str(current_dir / 'datasets/GH18_Gh19_embeddings.npy'), g18_g19_embeddings)